In [ ]:
# CELL 1 — Setup. Always safe to run. Run this first every session.
import pandas as pd
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")
print("Setup done. Key loaded:", api_key is not None)

In [ ]:
# CELL 2 — Load the prompts from disk (no regeneration, no API call)
prompts_df = pd.read_json("nab_prompts.json", orient="records")
print("Prompts loaded:", prompts_df.shape[0], "across", prompts_df["product_family"].nunique(), "families")

In [ ]:
# CELL 3 — Load the gather results from disk (no re-gathering, no API call)
results_df = pd.read_json("nab_credit_cards_results.json", orient="records")
print("Results loaded:", results_df.shape[0], "rows")

In [ ]:
# Inspect actual answers — trust the data before building analysis
import textwrap

for _, row in results_df.sort_values("model_requested").iterrows():
    print("=" * 80)
    print(f"{row['model_requested']}   (actual: {row['model_actual']})")
    print(f"Q: {row['question']}")
    print(f"mentioned={row['brand_mentioned']}   citations={row['num_citations']}")
    print("-" * 80)
    print(textwrap.fill(str(row['answer'])[:600], width=80))
    cites = row['citations'] if isinstance(row['citations'], list) else []
    if cites:
        print(f"\n  cites: {cites[:3]}")
    print()

In [ ]:
for model in results_df["model_requested"].unique():
    sub = results_df[results_df["model_requested"] == model]
    all_cites = [c for lst in sub["citations"] if isinstance(lst, list) for c in lst]
    sample = all_cites[0] if all_cites else "(none)"
    print(f"{model:35s} {len(all_cites):>3} cites | {sample[:70]}")

In [ ]:
import re

BRAND = re.compile(r'\bnab\b|national australia bank', re.IGNORECASE)

def mentions(text):
    text = str(text)
    return [text[max(0, m.start()-45):m.end()+45].replace("\n", " ")
            for m in BRAND.finditer(text)]

results_df["mentioned_v2"] = results_df["answer"].apply(lambda a: len(mentions(a)) > 0)

disagree = results_df[results_df["brand_mentioned"] != results_df["mentioned_v2"]]
print(f"Disagreements old vs new: {len(disagree)}")
print(disagree[["model_requested", "question", "brand_mentioned", "mentioned_v2"]].to_string())

print("\n--- every match in context ---")
for _, r in results_df.iterrows():
    hits = mentions(r["answer"])
    if hits:
        print(f"\n{r['model_requested']} | {r['question'][:50]}")
        for h in hits:
            print(f"   …{h}…")

In [ ]:
# one Gemini row's full raw payload — did search fire but annotations drop?
g = results_df[(results_df["model_requested"]=="google/gemini-3.7-flash:online") &
               (results_df["num_citations"]==0)].iloc[0]
print(g["question"])
print("keys:", list(g.index))
print("error:", g.get("error"))
# and check what you actually stored — do you keep the raw API response anywhere,
# or only the extracted citations list?

In [ ]:
import os, json, requests
from dotenv import load_dotenv
load_dotenv()

q = "I keep paying high interest on my credit card in Australia and want to find a better deal."

r = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"},
    json={"model": "google/gemini-3.7-flash:online",
          "messages": [{"role": "user", "content": q}]},
)
data = r.json()
print("status:", r.status_code)

# dump the full structure so nothing hides
print(json.dumps(data, indent=2)[:4000])

In [ ]:
results_df.shape

In [ ]:
results_df[results_df['num_citations'] > 0].shape[0]

In [ ]:
searched = results_df[results_df['num_citations'] > 0]
memory   = results_df[results_df['num_citations'] == 0]

In [ ]:
memory[['model_requested', 'brand_mentioned', 'question']]

In [ ]:
results_df[(results_df['brand_mentioned']) & (results_df['num_citations'] > 0)]

In [ ]:
from urllib.parse import urlparse

all_urls = results_df['citations'].explode().dropna()
print(f"{len(all_urls)} citation URLs across all 20 calls\n")
print(all_urls.value_counts().head(10))

In [ ]:
def get_domain(url):
    return urlparse(url).netloc.replace('www.', '')

domains = all_urls.apply(get_domain)
domains.value_counts().head(15)

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv()

# The schema you built — unchanged.
schema = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "description": "Every brand or organisation named in the answer.",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Name of the entity."},
                    "isCompetitor": {"type": "boolean", "description": "Whether the entity is a competitor to the brand being analysed."},
                    "sentiment": {
                        "type": "string",
                        "enum": ["positive", "neutral", "negative"],
                        "description": "Sentiment of the answer's coverage of this entity. Rate positive if brand is recommended highly or mentioned as the sole/most prominent example in a positive context. Rate neutral if they're listed/viable but not singled out in any way. Rate negative if they are criticised REGARDLESS of prominence."
                    }
                },
                "required": ["name", "isCompetitor", "sentiment"],
                "additionalProperties": False
            }
        }
    },
    "required": ["entities"],
    "additionalProperties": False
}


def analyse_answer(brand, answer):
    system_message = f"""You are analysing how an AI answer covers brands in a market.

The brand being analysed is: {brand}

Extract every brand or organisation named in the answer. For each one:
- name: the entity as named
- isCompetitor: true if it competes with {brand} in the same market, false otherwise (regulators, government bodies, and comparison/aggregator sites are not competitors)
- sentiment: how the answer treats that entity, per the definitions in the schema (if a brand is named first and alone as the recommendation, that is positive, not neutral. Only use neutral when a brand is one of several listed without preference.)

Only include entities actually named in the answer. Do not infer or add ones that aren't there."""

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"},
        json={
            "model": "openai/gpt-5.6-luna",
            "temperature": 0,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": answer},
            ],
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": "entity_analysis",
                    "strict": True,
                    "schema": schema,
                },
            },
        },
    )

    data = response.json()
    content = data["choices"][0]["message"]["content"]
    return json.loads(content)

In [ ]:
row = results_df.loc[2]
result = analyse_answer("NAB", row["answer"])
result